In [91]:
%pip install jupyter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [92]:
# Step 1: Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, classification_report

In [93]:
# Step 2: Load the dataset

data = pd.read_csv(r'C:/Users/Sakshi Agarwal/Downloads/Movie Genre/Indian_movies.csv',sep=',')

In [94]:
# Check the data format
print(data.head())

   Unnamed: 0  Rank                          Movie_Names  \
0           0     1  Ramayana: The Legend of Prince Rama   
1           1     2           Rocketry: The Nambi Effect   
2           2     3                              Nayakan   
3           3     4                             Gol Maal   
4           4     5                           Anbe Sivam   

                                               Links  Rating  Year  \
0  https://www.imdb.com//title/tt0259534/?ref_=fe...     9.2  1993   
1  https://www.imdb.com//title/tt9263550/?ref_=fe...     8.7  2022   
2  https://www.imdb.com//title/tt0093603/?ref_=fe...     8.6  1987   
3  https://www.imdb.com//title/tt0079221/?ref_=fe...     8.5  1979   
4  https://www.imdb.com//title/tt0367495/?ref_=fe...     8.6  2003   

  Duration_of_movie                                  Genere  \
0                PG  Animation,Action,Adventure,Back to top   
1            2h 37m             Biography,Drama,Back to top   
2         Not Rated          

In [95]:


# Get the names of all columns (these will just be integer indices)
column_names = data.columns.tolist()
print("\nNames of all columns:")
print(column_names)


Names of all columns:
['Unnamed: 0', 'Rank', 'Movie_Names', 'Links', 'Rating', 'Year', 'Duration_of_movie', 'Genere', 'Description']


In [96]:

import re

# Step 1: Remove duplicates
data = data.drop_duplicates(subset='Description', keep='first')

# Step 2: Handle missing values
# Remove rows where 'plot' or 'title' columns are missing
data = data.dropna(subset=['Description', 'Movie_Names'])

# Step 3: Preprocess the 'Description' text column
def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert text to lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply text cleaning to the 'plot' column
data['Description'] = data['Description'].apply(clean_text)

# Optional: Check the first few rows to ensure data is cleaned
print(data.head())

# The dataset is now cleaned and ready for further processing.


   Unnamed: 0  Rank                          Movie_Names  \
0           0     1  Ramayana: The Legend of Prince Rama   
1           1     2           Rocketry: The Nambi Effect   
2           2     3                              Nayakan   
3           3     4                             Gol Maal   
4           4     5                           Anbe Sivam   

                                               Links  Rating  Year  \
0  https://www.imdb.com//title/tt0259534/?ref_=fe...     9.2  1993   
1  https://www.imdb.com//title/tt9263550/?ref_=fe...     8.7  2022   
2  https://www.imdb.com//title/tt0093603/?ref_=fe...     8.6  1987   
3  https://www.imdb.com//title/tt0079221/?ref_=fe...     8.5  1979   
4  https://www.imdb.com//title/tt0367495/?ref_=fe...     8.6  2003   

  Duration_of_movie                                  Genere  \
0                PG  Animation,Action,Adventure,Back to top   
1            2h 37m             Biography,Drama,Back to top   
2         Not Rated          

In [97]:
# Step 3: Preprocess the text data
# Assuming the dataset has columns 'plot' (text) and 'genre' (labels)
# Split the 'genre' column into lists of genres for multi-label classification
data['Genere'] = data['Genere'].apply(lambda x: x.split(','))  # Split genres into lists

In [98]:

# Step 4: Convert text to TF-IDF features
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf_vectorizer.fit_transform(data['Movie_Names'])

In [99]:

# Step 5: Prepare the target labels (multi-label binarization)
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['Genere'])


In [100]:
# Step 6: Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [101]:

# Step 7: Train the Naive Bayes model with OneVsRest strategy for multi-label classification
model = OneVsRestClassifier(MultinomialNB(alpha=0.5))
model.fit(X_train, y_train)

c:\Users\Sakshi Agarwal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\multiclass.py:87: UserWarning: Label 3 is present in all training examples.
  warnings.warn(


OneVsRestClassifier(estimator=MultinomialNB(alpha=0.5))

In [102]:
# Step 8: Make predictions on the test set
y_pred = model.predict(X_test)

In [103]:
# Step 9: Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy*100:.2f}%')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=mlb.classes_))


Accuracy: 12.00%
Classification Report:
              precision    recall  f1-score   support

      Action       0.80      0.19      0.31        21
   Adventure       0.00      0.00      0.00         2
   Animation       0.00      0.00      0.00         0
 Back to top       1.00      1.00      1.00        50
   Biography       0.00      0.00      0.00         5
      Comedy       0.00      0.00      0.00         9
       Crime       1.00      0.17      0.29        18
       Drama       0.84      1.00      0.91        42
      Family       0.00      0.00      0.00         1
     Fantasy       0.00      0.00      0.00         1
     History       0.00      0.00      0.00         1
      Horror       0.00      0.00      0.00         0
       Music       0.00      0.00      0.00         2
     Musical       0.00      0.00      0.00         2
     Mystery       0.00      0.00      0.00         4
     Romance       0.00      0.00      0.00         6
      Sci-Fi       0.00      0.00      0.

c:\Users\Sakshi Agarwal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Sakshi Agarwal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Sakshi Agarwal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavio

In [104]:
# Step 10: Predict genres for a new movie plot
def predict_genre(Description):
    plot_transformed = tfidf_vectorizer.transform([Description])
    prediction = model.predict(plot_transformed)
    predicted_genres = mlb.inverse_transform(prediction)
    return predicted_genres[0] if predicted_genres else ['Unknown']





In [105]:
def predict_genre_by_name(Movie_Names):
     movie =tfidf_vectorizer.transform([Movie_Names])
     prediction = model.predict(movie)
     predicted_Genere = mlb.inverse_transform(prediction)
     return predicted_Genere[0] if predicted_Genere else['Unknown']

In [106]:
# Example usage of the prediction function
new_plot =" Four youngsters arrive in a big city and their lives become interlinked."
predicted_genres = predict_genre(new_plot)
print("Predicted genres:", predicted_genres)

Predicted genres: ('Back to top', 'Drama')


In [107]:
new_movie ="Gully Boy"
predicted_Genere = predict_genre_by_name(new_movie)
print("predicted genre:",predicted_Genere)

predicted genre: ('Back to top', 'Drama')
